# invest_panel_weo：合并与数据质量核验

## tl;dr

- 输出为 1,827 行、26 列、63 个国家/地区的 1995–2023 平衡面板。
- `iso3 + year` 无重复，合并没有改变基础面板行数。
- 新增 WEO GDP 列覆盖率：CurrentGDP 99.89%，ConstantGDP 99.89%，capitaGDP 99.78%；财政金额列覆盖率：revenue 96.93%，debt 94.36%。
- 派生列 interest_revenue 覆盖率为 92.28%，公式最大绝对误差为 1.42e-14。
- WEO 百分数保持原始百分比点单位，没有乘以 100。


## Context & Methods

本 notebook 是 CSV 与说明文档的审计附件。它重新读取基础面板、输出面板和 WEO 七个目标系列，检查字段保留、唯一键、左连接行数、WEO 数值一致性、派生公式、缺失率和描述统计。

### Key Assumptions

- `iso3 + year` 是目标面板唯一键。
- WEO `COUNTRY.ID` 与基础面板 `iso3` 可直接匹配。
- 分析期限定为 1995–2023。


## Data

### 1. Load inputs and output

In [ ]:
from pathlib import Path
import importlib.util
import sys
import pandas as pd

sys.dont_write_bytecode = True
OUTPUT_DIR = Path.cwd()
PROJECT_ROOT = OUTPUT_DIR.parent
BASE_CSV = PROJECT_ROOT / "cleaned_imf_like_panel_1995_2023.csv"
OUTPUT_CSV = OUTPUT_DIR / "invest_panel_weo.csv"

base = pd.read_csv(BASE_CSV)
panel = pd.read_csv(OUTPUT_CSV)

spec = importlib.util.spec_from_file_location("builder", OUTPUT_DIR / "build_invest_panel_weo.py")
builder = importlib.util.module_from_spec(spec)
spec.loader.exec_module(builder)
_, weo_numeric, weo_metadata, _ = builder.read_weo_values()

base.shape, panel.shape, weo_metadata.groupby("code")["iso3"].nunique().to_dict()


## Results

### 2. Validate grain and source-field preservation

In [ ]:
expected_columns = ["PrimaryBalance_gdp" if c == "OB_gdp" else c for c in base.columns] + list(builder.WEO_FIELDS.values()) + builder.DERIVED_FIELDS

renamed_base = base.rename(columns={"OB_gdp": "PrimaryBalance_gdp"})
preserved = renamed_base.equals(panel[renamed_base.columns])
checks = pd.Series({
    "output_rows": len(panel),
    "output_columns": panel.shape[1],
    "column_order_matches": panel.columns.tolist() == expected_columns,
    "base_values_preserved": preserved,
    "duplicate_iso3_year_keys": int(panel.duplicated(["iso3", "year"]).sum()),
    "exact_duplicate_rows": int(panel.duplicated().sum()),
    "countries": int(panel["iso3"].nunique()),
    "year_min": int(panel["year"].min()),
    "year_max": int(panel["year"].max()),
})
checks


### 3. Reconcile appended values to WEO

In [ ]:
weo_rows = [
    {"iso3": iso3, "year": year, "code": code, "source_value": value}
    for (iso3, year, code), value in weo_numeric.items()
]
weo_long = pd.DataFrame(weo_rows)
weo_wide = weo_long.pivot(index=["iso3", "year"], columns="code", values="source_value").reset_index()
weo_wide = weo_wide.rename(columns=builder.WEO_FIELDS)
reconciled = panel.merge(weo_wide, on=["iso3", "year"], how="left", suffixes=("_out", "_src"), validate="one_to_one")

reconciliation = {}
for column in builder.WEO_FIELDS.values():
    out = reconciled[f"{column}_out"]
    src = reconciled[f"{column}_src"]
    both = out.notna() & src.notna()
    reconciliation[column] = {
        "output_non_missing": int(out.notna().sum()),
        "source_non_missing": int(src.notna().sum()),
        "max_absolute_difference": float((out[both] - src[both]).abs().max()) if both.any() else None,
        "missingness_matches": bool(out.isna().equals(src.isna())),
    }
display(pd.DataFrame(reconciliation).T)

expected_interest = (panel["PrimaryBalance_gdp"] - panel["OverallBalance_gdp"]) / panel["Revenue_gdp"] * 100
expected_interest_mask = panel[["PrimaryBalance_gdp", "OverallBalance_gdp", "Revenue_gdp"]].notna().all(axis=1) & panel["Revenue_gdp"].ne(0)
interest_check = pd.Series({
    "output_non_missing": int(panel["interest_revenue"].notna().sum()),
    "expected_non_missing": int(expected_interest_mask.sum()),
    "missingness_matches": bool(panel["interest_revenue"].notna().equals(expected_interest_mask)),
    "max_absolute_difference": float((panel.loc[expected_interest_mask, "interest_revenue"] - expected_interest[expected_interest_mask]).abs().max()),
    "zero_revenue_gdp_rows": int(panel["Revenue_gdp"].eq(0).sum()),
})
interest_check


### 4. Review completeness

In [ ]:
coverage = pd.DataFrame({
    "non_missing": panel.notna().sum(),
    "missing": panel.isna().sum(),
    "coverage_pct": panel.notna().mean().mul(100),
})
coverage


### 5. Review numeric distributions

In [ ]:
panel.describe(percentiles=[0.25, 0.5, 0.75]).T


## Takeaways

- 面板键和行数检查通过，基础字段在改名后逐值保持一致。
- 七个 WEO 字段与源值及缺失位置一致，未发生单位缩放。
- `interest_revenue` 严格按 `((PrimaryBalance_gdp - OverallBalance_gdp) / Revenue_gdp) * 100` 计算，单位为百分数。
- 财政收入和总体余额缺失主要发生在样本早期；建模时应记录最终可用样本。
- CurrentGDP、ConstantGDP、revenue 和 debt 为十亿本币，不适合未经汇率或 PPP 转换的跨国水平比较；capitaGDP 为固定价格 PPP 国际元/人。
